# Tools

### Models can request to call tool that perform task such as fetching data from a databse, or searching the web or running code. Tools are pairing of:

1. A schema, including the name of tool, a description, and/or argument definition
2. A function or coroutine to execute

In [9]:
# ChatGROQ
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model(model="groq:qwen/qwen3.6-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talk?" This is a common question about animal behavior, specifically parrot vocalization/mimicry.\n\n2.  **Identify Key Concepts**:\n   - Parrots don\'t actually "talk" like humans (they don\'t understand language in the human sense)\n   - They mimic sounds\n   - Evolutionary/social reasons for mimicry\n   - Brain structure (vocal learning centers)\n   - Social bonding, communication within flocks\n   - Captivity vs. wild behavior\n   - Scientific understanding of avian vocal learning\n\n3.  **Structure the Answer**:\n   - Clarify the misconception (they mimic, don\'t truly "talk")\n   - Explain the biological/neurological basis\n   - Explain the evolutionary/social reasons\n   - Differentiate wild vs. captive behavior\n   - Mention what they actually understand (if anything)\n   - Keep it clear, scientific but accessible\n\n4.  **Draft - Section by Section**:\n 

In [10]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    return f"It is Sunny in {location}"

model_with_tool = model.bind_tools([get_current_weather])

In [11]:
response = model_with_tool.invoke("What is the current weather in College Station?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool Name: {tool_call["name"]}")
    print(f"Tool Arguments: {tool_call["args"]}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to know the current weather in "College Station".\n2.  **Identify Available Tools**: `get_current_weather(location: string)`\n3.  **Map Intent to Tool**: The tool `get_current_weather` matches the intent. The parameter `location` should be "College Station".\n4.  **Execute Tool Call**: `get_current_weather(location="College Station")`\n5.  **Format Output**: Ensure correct JSON structure for tool calling. No extra text. Return the tool call.✅\n', 'tool_calls': [{'id': 'ac6scykx8', 'function': {'arguments': '{"location":"College Station"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 280, 'total_tokens': 432, 'completion_time': 0.288878296, 'completion_tokens_details': {'reasoning_tokens': 122}, 'prompt_time': 0.020840916, 'prompt_tokens_details': None, 'queue_time': 0.020521341, 'total

### Tool Execution Loops

In [12]:
#Step 1 : Model generates tool call.

messages = [{"role": "user", "content": "What is the current weather in College Station?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

#Step 2 : Execute tool and collect tool output.
for tool_call in ai_msg.tool_calls:
    tool_result = get_current_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass the result back to the model for final response.
final_response = model_with_tool.invoke(messages)
print(final_response.text)

It is currently sunny in College Station.


In [13]:
messages

[{'role': 'user',
  'content': 'What is the current weather in College Station?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to know the current weather in "College Station".\n2.  **Identify Available Tools**: I have `get_current_weather` which takes a `location` parameter.\n3.  **Map Parameters**: `location` = "College Station".\n4.  **Execute Tool Call**: Call `get_current_weather(location="College Station")`.\n5.  **Format Output**: Return the tool call in the specified format.✅\n', 'tool_calls': [{'id': 'ewe327nw8', 'function': {'arguments': '{"location":"College Station"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens': 280, 'total_tokens': 416, 'completion_time': 0.286994071, 'completion_tokens_details': {'reasoning_tokens': 106}, 'prompt_time': 0.023844886, 'prompt_tokens_details': None, 'queue_time': 0.032445